Implementar re-ranking en el retrieval.
-   Probar re-rank en RAG
-   Probar re-rank en grafo
-   Probar re-rank Combinado

INICIALIZACION

In [1]:
import sys
from sentence_transformers import SentenceTransformer

c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append("../src")

In [3]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j
from graph_retrieval.funciones_graph_retrieval import (
    extraer_top_k_entities,
    formatear_tripletas,
)
from conexion_qdrant.conexion_qdrant import ConexionQdrant
from RAG_retrieval.funciones_RAG_retrieval import(
    extraer_info_nodes,
    extraer_info_points,
    filtrar_nodos_por_score,
    textos_para_prompt
)
from funciones_generales import build_prompt
from LLM_interaction import LLM_interaction_functions as llm_funcs
from metricas.metricas_2Wiki import (
    f1_score,
    exact_match_score,
    respuesta_en_nodos_encontrados,
    suporting_facts_en_subgrafo,
    metricas_totales
    )
from output_save.funciones_guardado import guardar_resultados, guardar_registro

# Load Data

In [4]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 100, "train")

In [5]:
ejemplo = dataset_2Wiki[0]

# Qdrant y Neo4j conexion

In [6]:
database_Neo = "2wiki.prueba1"
database_Neo4j = ConexionNeo4j(database_Neo)
qd_client = ConexionQdrant()

In [7]:
collection = "2wikimultihop_prueba1"
embed_model_st = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4740.54it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Extraer entidades (spacy)

In [8]:
dataset_2Wiki[0]

{'_id': '13f5ad2c088c11ebbd6fac1f6bf848b6',
 'type': 'bridge_comparison',
 'question': 'Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?',
 'context': '[["Stuart Rosenberg", ["Stuart Rosenberg (August 11, 1927 \\u2013 March 15, 2007) was an American film and television director whose motion pictures include \\"Cool Hand Luke\\" (1967), \\"Voyage of the Damned\\" (1976), \\"The Amityville Horror\\" (1979), and \\"The Pope of Greenwich Village\\" (1984).", "He was noted for his work with actor Paul Newman."]], ["M\\u00e9diterran\\u00e9e (1963 film)", ["M\\u00e9diterran\\u00e9e is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schl\\u00f6ndorff.", "It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.", "The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Go

In [9]:
import spacy

1º descargar modelo

In [ ]:
# import subprocess

# subprocess.run("python -m spacy download en_core_web_sm")

In [12]:
ner = spacy.load('en_core_web_sm')

In [17]:
question = ejemplo["question"]
question

'Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?'

In [18]:
entidades = ner(question)

In [22]:
entidades.ents

(1970, Méditerranée, 1963)

In [ ]:
print(entidades.ents[1].text)
print(entidades.ents[1].label_)
print(entidades.ents[1].start_char)
print(entidades.ents[1].end_char)
print(entidades.ents[1].sent.text)

In [24]:

question = dataset_2Wiki[1]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Do both films The Falcon (Film) and Valentin The Good have the directors from the same country?
(Valentin The Good,)


In [25]:
question = dataset_2Wiki[2]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Which film whose director is younger, Charge It To Me or Danger: Diabolik?
()


In [26]:
question = dataset_2Wiki[3]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

What is the date of birth of Mina Gerhardsen's father?
(Mina Gerhardsen's,)


In [27]:
question = dataset_2Wiki[4]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

What nationality is the director of film Wedding Night In Paradise (1950 Film)?
(film Wedding Night, 1950)


Al NER de spacy se le pueden agregar etapas a la hora de extraer entidades.  

Realmente lo que hace es pasar varias etapas (pipeline) hasta que las detecta.  

Se pueden añadir etapas antes o despues.  
Hay formas predeterminadas para que no deje pasar nombres que empiecen por mayuscula.

In [28]:
pattern = [
    {
        "IS_TITLE": True,  # Detecta entidades cuando empiezan por mayuscula
        "OP": "+"          # Se repite 1 o más veces
    }
]

In [34]:
if "entity_ruler" in ner.pipe_names:
    ner.remove_pipe("entity_ruler")
ner_custom = ner.add_pipe("entity_ruler", before="ner")

# Patrón: Cualquier palabra que empiece con mayúscula, repetida 1 o más veces
# patterns = [
#     {
#         "label": "MOVIE", 
#         "pattern": [
#             {
#                 "TEXT": {"REGEX": "^[A-Z][a-zA-Z0-9]*$"}, 
#                 "OP": "+"
#             }
#         ]
#     }
# ]
patron_mayuscula = [
    {
        "IS_TITLE": True,  # Detecta entidades cuando empiezan por mayuscula
        "OP": "+"          # Se repite 1 o más veces
    }
]
patterns = [
    {
        "label": "MOVIE",          # El nombre de la entidad que quieres asignar
        "pattern": patron_mayuscula  # El patrón que definiste arriba
    }
]


ner_custom.add_patterns(patterns)

In [35]:
question = dataset_2Wiki[0]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?
(Are, Move, 1970, Film, Méditerranée, 1963, Film)


In [36]:
question = dataset_2Wiki[1]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Do both films The Falcon (Film) and Valentin The Good have the directors from the same country?
(Do, The Falcon, Film, Valentin The Good)


In [37]:
question = dataset_2Wiki[2]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Which film whose director is younger, Charge It To Me or Danger: Diabolik?
(Which, Charge It To Me, Danger, Diabolik)


In [38]:
question = dataset_2Wiki[3]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

What is the date of birth of Mina Gerhardsen's father?
(What, Mina Gerhardsen)


In [39]:
question = dataset_2Wiki[4]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

What nationality is the director of film Wedding Night In Paradise (1950 Film)?
(What, Wedding Night In Paradise, 1950, Film)


In [40]:
question = dataset_2Wiki[5]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

When is the composer of film Sruthilayalu 's birthday?
(When, Sruthilayalu)


In [41]:
question = dataset_2Wiki[6]["question"]
print(question)
entidades = ner(question)
print(entidades.ents)

Who is Rhescuporis I (Odrysian)'s paternal grandfather?
(Who, Rhescuporis I, Odrysian)
